### Imports

In [ ]:
import pickle
from copy import copy
from tqdm import tqdm

import re
from typing import List, Tuple, Generator

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.normalizers import Lowercase

from transformers import PreTrainedTokenizerFast

c:\main\GitHub\nlpTasks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Dataset loading

In [2]:
with open("data\\pos_data.txt", 'r', encoding = "utf-8") as f:
    data = list(filter(None, [line.rstrip("\n") for line in f]))

In [3]:
def merge_sentences(data: List[str]) -> List[List[Tuple[str, str]]]:
    """Merges sentences from the given list of lines"""
    sentences = []
    current_sentence = []
    for row in data:
        word_tag = re.search("\t.*?\t", row)[0].strip("\t")
        pos = re.search("[A-z]+", row)[0].strip("|") if word_tag not in ["<end>", "<beg>"] else re.search("\t.*?\t", row)[0].strip("\t")
        current_sentence.append((word_tag, pos))
        if current_sentence[0][0] == "<beg>" and current_sentence[-1][0] == "<end>":
            sentences.append(copy(current_sentence))
            current_sentence.clear()
        else:
            continue
    return sentences


In [4]:
processed_sentences = merge_sentences(data)
processed_sentences

[[('<beg>', '<beg>'),
  ('рой', 'VERB'),
  ('колодец', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'), ('рой', 'VERB'), ('погреб', 'NOUN'), ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('рой', 'VERB'),
  ('укрытие', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('я', 'PRON'),
  ('мою', 'VERB'),
  ('окно', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('сотрудники', 'NOUN'),
  ('милиции', 'NOUN'),
  ('вечером', 'NOUN'),
  ('31', 'NUM'),
  ('декабря', 'NOUN'),
  ('уничтожили', 'VERB'),
  ('в', 'ADP'),
  ('хасавюрте', 'NOUN'),
  ('четверых', 'NUM'),
  ('боевиков', 'NOUN'),
  ('.', 'PUNCT'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('вместе', 'ADV'),
  ('с', 'ADP'),
  ('ним', 'PRON'),
  ('берлускони', 'NOUN'),
  ('выпустил', 'VERB'),
  ('уже', 'ADV'),
  ('четыре', 'NUM'),
  ('пластинки', 'NOUN'),
  ('.', 'PUNCT'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('психически', 'ADV'),
  ('неуравновешенный', 'ADJ'),
  ('житель', 'NOUN'),
  ('милана', 'NOUN'),
 

In [5]:
# Saved processed dataset
# with open('artifacts\\pos_processed_dataset.pkl', 'wb') as file:
#     pickle.dump(processed_sentences, file)

In [2]:
# Load processed dataset
with open('artifacts\\pos_processed_dataset.pkl', 'rb') as file:
    processed_dataset = pickle.load(file)

In [3]:
processed_dataset

[[('<beg>', '<beg>'),
  ('рой', 'VERB'),
  ('колодец', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'), ('рой', 'VERB'), ('погреб', 'NOUN'), ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('рой', 'VERB'),
  ('укрытие', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('я', 'PRON'),
  ('мою', 'VERB'),
  ('окно', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('сотрудники', 'NOUN'),
  ('милиции', 'NOUN'),
  ('вечером', 'NOUN'),
  ('31', 'NUM'),
  ('декабря', 'NOUN'),
  ('уничтожили', 'VERB'),
  ('в', 'ADP'),
  ('хасавюрте', 'NOUN'),
  ('четверых', 'NUM'),
  ('боевиков', 'NOUN'),
  ('.', 'PUNCT'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('вместе', 'ADV'),
  ('с', 'ADP'),
  ('ним', 'PRON'),
  ('берлускони', 'NOUN'),
  ('выпустил', 'VERB'),
  ('уже', 'ADV'),
  ('четыре', 'NUM'),
  ('пластинки', 'NOUN'),
  ('.', 'PUNCT'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('психически', 'ADV'),
  ('неуравновешенный', 'ADJ'),
  ('житель', 'NOUN'),
  ('милана', 'NOUN'),
 

In [9]:
words = [item[0] for sentence_list in processed_dataset for item in sentence_list]
print(len(words))
words

10251201


['<beg>',
 'рой',
 'колодец',
 '<end>',
 '<beg>',
 'рой',
 'погреб',
 '<end>',
 '<beg>',
 'рой',
 'укрытие',
 '<end>',
 '<beg>',
 'я',
 'мою',
 'окно',
 '<end>',
 '<beg>',
 'сотрудники',
 'милиции',
 'вечером',
 '31',
 'декабря',
 'уничтожили',
 'в',
 'хасавюрте',
 'четверых',
 'боевиков',
 '.',
 '<end>',
 '<beg>',
 'вместе',
 'с',
 'ним',
 'берлускони',
 'выпустил',
 'уже',
 'четыре',
 'пластинки',
 '.',
 '<end>',
 '<beg>',
 'психически',
 'неуравновешенный',
 'житель',
 'милана',
 'швырнул',
 'в',
 'политика',
 'тяжелую',
 'сувенирную',
 'статуэтку',
 '.',
 '<end>',
 '<beg>',
 'в',
 'марте',
 '2009',
 'года',
 'литва',
 'подписала',
 'десятилетний',
 'контракт',
 'на',
 'поставки',
 'электроэнергии',
 'из',
 'россии',
 '.',
 '<end>',
 '<beg>',
 'атмосферу',
 'на',
 'улицах',
 'сотрудники',
 'штаба',
 'охарактеризовали',
 'как',
 'праздничную',
 '.',
 '<end>',
 '<beg>',
 'бельгиец',
 'будет',
 'занимать',
 'пост',
 'президента',
 'ес',
 'два',
 'с',
 'половиной',
 'года',
 '.',
 '<end

In [10]:
words.extend(["<pad>", "<unk>"])

In [11]:
main_tags = set(item[1] for sentence_list in processed_dataset for item in sentence_list)
main_tags

{'dior',
 'lada',
 'chemie',
 'guros',
 'sa',
 'wba',
 'tribal',
 'visa',
 'galli',
 'saudi',
 'mezzo',
 'combat',
 'casa',
 'defence',
 'wikislakis',
 'skrillex',
 'ta',
 'grey',
 'lxg',
 'reynders',
 'hn',
 'engadget',
 'bye',
 'sapphires',
 'tez',
 'ambassde',
 'weibo',
 'consis',
 'mag',
 'engineering',
 'euglena',
 'ingolstadt',
 'avantime',
 'christianus',
 'cp',
 'shift',
 'instruments',
 'jour',
 'gdb',
 'research',
 'mukherjee',
 'sicula',
 'des',
 'nix',
 'russie',
 'gbi',
 'ala',
 'intermezzo',
 'didier',
 'alike',
 'eu',
 'sphinx',
 'reckitt',
 'processing',
 'b',
 'fmcg',
 'gamestop',
 'sun',
 'essenze',
 'lz',
 'kommissia',
 'comscore',
 'msn',
 'nafta',
 'tertiary',
 'blossom',
 'cash',
 'lcs',
 'winapi',
 'your',
 'cadbury',
 'bocharov',
 'bt',
 'small',
 'nm',
 'sikorski',
 'martell',
 'australasia',
 'rating',
 'croma',
 'working',
 'ems',
 'amsbook',
 'beurteilen',
 'dancing',
 'pokemon',
 'kl',
 'nedcar',
 'unilever',
 'boomerang',
 'vitraux',
 'social',
 'tournamen

In [12]:
print(len(main_tags))

5206


In [13]:
# Create dict for class labels
key = 0
label_dict = dict()
for tag in main_tags:
    label_dict[tag] = key
    key += 1

In [14]:
label_dict["NOUN"]

1324

### Text tokenization

In [15]:
tokenizer = Tokenizer(model = BPE(unk_token = "<unk>"))
tokenizer.normalizer = Lowercase()

trainer = BpeTrainer(vocab_size = 30_000, special_tokens = ["<pad>", "<beg>", "<end>", "<unk>"])

In [16]:
tokenizer.train_from_iterator(iterator=words, trainer = trainer)

In [17]:
base_vocabulary = tokenizer.get_vocab()
base_vocabulary["<unk>"]
base_vocabulary

{'боялись': 9987,
 'мужик': 7040,
 'образования': 3723,
 'ником': 2386,
 'герма': 3618,
 'уголовных': 22490,
 'эми': 9353,
 'ринулся': 29461,
 'первом': 5515,
 'исход': 7497,
 '62': 9234,
 'познакомился': 15168,
 'различи': 22468,
 'коридору': 12267,
 'зме': 5380,
 'ти': 542,
 'кончено': 14578,
 'упрям': 23102,
 'чать': 2321,
 'здеш': 10272,
 'пере': 628,
 'гнет': 20742,
 'глазах': 2188,
 'структурой': 29637,
 'искази': 19113,
 'стене': 5327,
 'амби': 15688,
 'ценности': 9165,
 'столом': 5312,
 'темпами': 15856,
 'постепенно': 3381,
 'сати': 7917,
 'голь': 7594,
 'стихии': 26315,
 'подобно': 11139,
 'пресси': 11263,
 'приезжай': 23810,
 'челове': 767,
 'поляну': 26703,
 'криком': 16568,
 'рассказываю': 22257,
 'крис': 15864,
 'ваемые': 15331,
 'иван': 4411,
 'составлять': 27471,
 'му': 565,
 'вни': 2083,
 'слышим': 21703,
 'словах': 7821,
 'вызова': 23052,
 'фа': 874,
 'рекламы': 13071,
 'все же': 2521,
 'знакомого': 25238,
 'времени': 1406,
 'споко': 2265,
 'факторов': 14945,
 'коротк

In [18]:
# calculate maximum length of sentences
max_length = 0
for sentence in processed_dataset:
    max_length = max(max_length, len(sentence))
max_length

472

In [19]:
main_tokenizer = PreTrainedTokenizerFast(tokenizer_object = tokenizer, pad_token = "<pad>")

In [20]:
encoded_sentence = main_tokenizer("Хорошеесообщение", padding="max_length", max_length=max_length)["input_ids"]
encoded_sentence

[6761,
 7102,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0

In [21]:
main_tokenizer.decode(encoded_sentence, skip_special_tokens = True)

'хорошее сообщение'

In [22]:
# calculate maximum length of sentences
main_tokenizer

TokenizersBackend(name_or_path='', vocab_size=30000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<beg>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<end>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

### Dataset preparation

In [23]:
processed_dataset

[[('<beg>', '<beg>'),
  ('рой', 'VERB'),
  ('колодец', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'), ('рой', 'VERB'), ('погреб', 'NOUN'), ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('рой', 'VERB'),
  ('укрытие', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('я', 'PRON'),
  ('мою', 'VERB'),
  ('окно', 'NOUN'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('сотрудники', 'NOUN'),
  ('милиции', 'NOUN'),
  ('вечером', 'NOUN'),
  ('31', 'NUM'),
  ('декабря', 'NOUN'),
  ('уничтожили', 'VERB'),
  ('в', 'ADP'),
  ('хасавюрте', 'NOUN'),
  ('четверых', 'NUM'),
  ('боевиков', 'NOUN'),
  ('.', 'PUNCT'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('вместе', 'ADV'),
  ('с', 'ADP'),
  ('ним', 'PRON'),
  ('берлускони', 'NOUN'),
  ('выпустил', 'VERB'),
  ('уже', 'ADV'),
  ('четыре', 'NUM'),
  ('пластинки', 'NOUN'),
  ('.', 'PUNCT'),
  ('<end>', '<end>')],
 [('<beg>', '<beg>'),
  ('психически', 'ADV'),
  ('неуравновешенный', 'ADJ'),
  ('житель', 'NOUN'),
  ('милана', 'NOUN'),
 

In [33]:
# Create generator for dataset
def dataset_generator(tokenizer = main_tokenizer):
        for sentence in processed_dataset:
            word_sentence = []
            targets = []
            for tup in sentence:
                word_sentence.append(tup[0])
                tokenized_word_length = len(tokenizer(tup[0])["input_ids"])
                if tokenized_word_length > 1:
                    for _ in range(tokenized_word_length):
                        targets.append(tup[1])
                else:
                    targets.append(tup[1])
            full_sentence = []
            for word in word_sentence:
                full_sentence.extend(tokenizer(word)["input_ids"])
            yield full_sentence, [label_dict[target] for target in targets]

In [35]:
processed_dataset_generator = dataset_generator()

Going through dataset:   0%|          | 6/990066 [03:04<8461:15:28, 30.77s/it]


In [36]:
for tokenized_sent, tokenized_tgt in tqdm(processed_dataset_generator, desc="Going through generator", total = 990066):
    assert len(tokenized_sent) == len(tokenized_tgt),  "Allignment problem"

Going through generator: 100%|██████████| 990066/990066 [07:57<00:00, 2073.23it/s]


### Model definition

In [43]:
len(main_tokenizer.get_vocab())

30000

In [ ]:
class POSModel(nn.Module):
    def __init__(self, input_size:int, hidden_size:int,
                 num_embeddings:int = len(main_tokenizer.get_vocab()),
                 bidirectional: bool = False, vocab_size: int = len(main_tokenizer.get_vocab())):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, input_size)
        self.rnn = nn.LSTM(input_size, hidden_size, bidirectional = bidirectional)
        self.linear = nn.Linear(2*hidden_size, vocab_size)
        
    def forward(self, x):
        y = self.embedding(x)
        y = y.transpose(0, 1).squeeze(dim = 1)
        out, h = self.rnn(y) # S x B x H
        return out
        y = self.linear(out)
        y = y.flatten(0, 1)
        return y

In [70]:
model = POSModel(input_size = 300, hidden_size = 300, num_embeddings=30_000, bidirectional = True)
model

POSModel(
  (embedding): Embedding(30000, 300)
  (rnn): LSTM(300, 300, bidirectional=True)
  (linear): Linear(in_features=600, out_features=30000, bias=True)
)

### Training

In [71]:
epoch = 1_0000

In [72]:
def train_model(dataset, model):
    for ep in range(epoch):
        for data, tgt in dataset:
            data, tgt = torch.tensor(data).reshape(1, -1), torch.tensor(tgt).reshape(1, -1)
            prediction = model(data)
            return prediction

In [73]:
result = train_model(dataset_generator(), model)

In [74]:
result.shape

torch.Size([4, 1, 600])

In [283]:
training_sample = next(iter(train_dataloader))
print(f"Training sample shape: {training_sample[0].shape}")
training_sample

Training sample shape: torch.Size([32, 472])


[tensor([[    1,   629,  1335,  ...,     0,     0,     0],
         [    1,  7400,  5930,  ...,     0,     0,     0],
         [    1,   151,   828,  ...,     0,     0,     0],
         ...,
         [    1,  1881,  1917,  ...,     0,     0,     0],
         [    1,   568,    16,  ...,     0,     0,     0],
         [    1,   151, 26825,  ...,     0,     0,     0]]),
 tensor([[1974, 4725,  340,  ..., 4136, 4136, 4136],
         [1974, 1155, 2073,  ..., 4136, 4136, 4136],
         [1974, 4322,  340,  ..., 4136, 4136, 4136],
         ...,
         [1974, 1155, 2073,  ..., 4136, 4136, 4136],
         [1974, 4725, 4071,  ..., 4136, 4136, 4136],
         [1974, 4322,  340,  ..., 4136, 4136, 4136]]),
 tensor([ 8,  7, 12, 11,  6,  8, 13,  9, 12,  6,  9, 13,  9, 22,  6,  8, 17,  6,
         20,  8, 15, 11,  9, 11, 17,  5,  9, 18, 11,  6, 12,  8])]

In [292]:
model = POSModel(500, 300).to(device = 'cuda')
model(training_sample[0].to(device = 'cuda')).shape

torch.Size([15104, 30000])

In [293]:
training_sample[1].reshape(-1).shape

torch.Size([15104])

In [298]:
# Training loop for model_training
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    save: bool = False,
    save_epoch: int = 0,
    base_path: str = '',
    save_model_name: str = '',
    save_optimizer_name : str = ''
) -> float:
    """Run one training epoch."""
    if save:
        torch.save(model.state_dict(), base_path + f"Epoch #{save_epoch}" + "-" + save_model_name)
        torch.save(optimizer.state_dict(), base_path + f"Epoch #{save_epoch}" + "-" + save_optimizer_name)
    model.train()
    total_loss = 0

    for inputs, targets, lengths in tqdm(dataloader, desc="Epoch training"):
        inputs, targets, lengths = inputs.to(device), targets.to(device), lengths.to(device = 'cpu')
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, targets.reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        print(loss.item())

    return total_loss / len(dataloader)

In [299]:
train_dataloader = DataLoader(dataset = pos_dataset, shuffle = True, batch_size = 32)
train_dataloader

In [300]:
optimizer = optim.Adam(params=model.parameters(), lr = 0.005, betas = (0.99, 0.98))
criterion = nn.CrossEntropyLoss(ignore_index = 4136)

In [301]:
loss_result = train_one_epoch(model,
                              train_dataloader,
                              optimizer,
                              criterion,
                              device = 'cuda')
loss_result

Epoch training:   0%|          | 1/30940 [00:05<51:23:06,  5.98s/it]

2.332825183868408


Epoch training:   0%|          | 2/30940 [00:12<55:31:39,  6.46s/it]

2.729342222213745


Epoch training:   0%|          | 3/30940 [00:19<57:23:22,  6.68s/it]

...

2.3363475799560547


Epoch training:   1%|          | 294/30940 [33:40<58:46:11,  6.90s/it]

2.4682672023773193


Epoch training:   1%|          | 294/30940 [33:47<58:42:33,  6.90s/it]


KeyboardInterrupt: 

### Inference